# Loan Dataset Exploration with PySpark

This notebook is set up for the large `data/loan.csv` file using a local Spark session.

Prerequisites:
- Activate the virtual environment in the project root.
- Install the packages from `requirements.txt`.
- Install a Java runtime, since PySpark needs Java to start.


In [1]:
import os
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JAVA_HOME = Path('/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home')
os.environ.setdefault('JAVA_HOME', str(JAVA_HOME))
os.environ['PATH'] = f"{JAVA_HOME / 'bin'}:{os.environ['PATH']}"
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
CSV_PATH = DATA_DIR / 'loan.csv'
DICT_PATH = DATA_DIR / 'LCDataDictionary.xlsx'

spark = (
    SparkSession.builder
    .appName('loan-exploration')
    .master('local[*]')
    .config('spark.sql.repl.eagerEval.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'CSV exists: {CSV_PATH.exists()} | size: {CSV_PATH.stat().st_size / (1024 ** 3):.2f} GB')
print(f'Data dictionary exists: {DICT_PATH.exists()}')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 17:50:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
CSV exists: True | size: 1.11 GB
Data dictionary exists: True


In [ ]:
dictionary_preview = pd.read_excel(DICT_PATH).head(10)
dictionary_preview

In [ ]:
df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(CSV_PATH))
)

df.cache()
row_count = df.count()
column_count = len(df.columns)
print(f'Rows: {row_count:,}')
print(f'Columns: {column_count}')


In [ ]:
df.printSchema()

In [ ]:
df.select(df.columns[:12]).show(10, truncate=False)

In [ ]:
missing_summary = (
    df.select([
        F.sum(F.col(c).isNull().cast('int')).alias(c)
        for c in df.columns
    ])
    .toPandas()
    .T
    .reset_index()
)

missing_summary.columns = ['column', 'missing_rows']
missing_summary['missing_pct'] = missing_summary['missing_rows'] / row_count
missing_summary.sort_values('missing_pct', ascending=False).head(20)

In [ ]:
candidate_targets = [c for c in ['loan_status', 'grade', 'sub_grade', 'purpose', 'home_ownership'] if c in df.columns]
candidate_targets

In [ ]:
if 'loan_status' in df.columns:
    (
        df.groupBy('loan_status')
        .count()
        .orderBy(F.desc('count'))
        .show(20, truncate=False)
    )
else:
    print('`loan_status` is not present in the dataset.')


In [ ]:
numeric_candidates = [
    c for c, t in df.dtypes
    if t in {'int', 'bigint', 'float', 'double', 'decimal'}
]

df.select(numeric_candidates[:15]).summary().show(truncate=False)

## Next Step

If the schema looks reasonable, the next practical move is to persist the raw CSV as Parquet for faster repeated analysis:

```python
parquet_path = DATA_DIR / 'loan.parquet'
df.write.mode('overwrite').parquet(str(parquet_path))
```
